## Exercícios

> Retirados de [learn-python: sqlalchemy_orm-questions](https://aviadr1.github.io/learn-advanced-python/11_db_access/exercise/sqlalchemy_orm-questions.html).

#### Q1.

Baixa e extraia o arquivo compactado com o banco de dados [Chinook database](https://www.sqlitetutorial.net/sqlite-sample-database/). Salve o arquivo `chinook.db` na mesma pasta deste script.
* Link para baixar: http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip

<img width=500 src=https://www.sqlitetutorial.net/wp-content/uploads/2015/11/sqlite-sample-database-color.jpg>


#### Q2.

Leia o código e os comentários das células a seguir para entender como acessamos os modelos ORM de um banco já existente.

In [ ]:
from sqlalchemy import create_engine, text, MetaData
from sqlalchemy.orm import Session

engine = create_engine("sqlite+pysqlite:///chinook.db", echo=False)

### extrai as classes da base de dados Chinook
metadata = MetaData()
metadata.reflect(engine)

# O metadata tem informações sobre as tabelas
# que serão usadas para criar os modelos ORM
for table_name, table in metadata.tables.items():
    print(table_name)
    print(table.columns.keys())
    print(table.columns.items())
    print('-'*25)

### configura o objeto Base mapeando os modelos ORM das tabelas
from sqlalchemy.ext.automap import automap_base
Base = automap_base(metadata=metadata)
Base.prepare()

# o objeto Base tem os modelos ORM que podemos usar
# para manipular o banco de dados
print(Base.classes.items())

In [ ]:
# A seguir um exemplo de query na tabela Albums
# usamos o objeto Base para acessar cada modelo ORM.

session = Session(engine)
res = session.scalars(select(Base.classes.albums))
first_album = res.first()
print(first_album.AlbumId, first_album.Title)

#### Q3. 
Com base nos códigos anteriores realize as operações solicitadas nas células a seguir:


In [ ]:
from sqlalchemy import select, func

### Imprima os três primeiros registros da tabela tracks

print("--- 3 primeiros registros da tabela tracks ---")
stmt1 = select(Tracks).limit(3)
for track in session.scalars(stmt1):
    print(f"ID: {track.TrackId} | Nome: {track.Name} | Compositor: {track.Composer}")

print("\n" + "="*50 + "\n")

In [ ]:
### Imprima o nome da faixa e o título do álbum das primeiras 20 faixas na tabela tracks.
print("--- Nome da faixa e título do álbum (Primeiras 20) ---")
stmt2 = select(Tracks, Albums).join(Albums, Tracks.AlbumId == Albums.AlbumId).limit(20)
for track, album in session.execute(stmt2):
    print(f"Faixa: {track.Name} | Álbum: {album.Title}")

print("\n" + "="*50 + "\n")

In [ ]:
### Imprima as 10 primeiras vendas de faixas da tabela invoice_items
print("--- 10 primeiras vendas da tabela invoice_items ---")
stmt3 = select(InvoiceItems).limit(10)
for item in session.scalars(stmt3):
    print(f"InvoiceLineId: {item.InvoiceLineId} | InvoiceId: {item.InvoiceId} | TrackId: {item.TrackId}")

print("\n" + "="*50 + "\n")
### Para essas 10 primeiras vendas, imprima os nomes das faixas vendidas e a quantidade vendida.
print("--- Nomes das faixas e quantidades das 10 primeiras vendas ---")
stmt4 = select(InvoiceItems, Tracks).join(Tracks, InvoiceItems.TrackId == Tracks.TrackId).limit(10)
for item, track in session.execute(stmt4):
    print(f"Faixa: {track.Name} | Qtd: {item.Quantity}")

session.close()

In [ ]:
session = Session(engine)

# Referenciando a classe de artistas restante
Artists = Base.classes.artists

### Imprima os nomes das 10 faixas mais vendidas e quantas vezes foram vendidas.
print("--- Top 10 Faixas Mais Vendidas ---")
stmt5 = (
    select(Tracks.Name, func.sum(InvoiceItems.Quantity).label('total_vendido'))
    .join(InvoiceItems, Tracks.TrackId == InvoiceItems.TrackId)
    .group_by(Tracks.TrackId, Tracks.Name)
    .order_by(func.sum(InvoiceItems.Quantity).desc())
    .limit(10)
)

for nome_faixa, total in session.execute(stmt5):
    print(f"Vendas: {total} | Faixa: {nome_faixa}")

print("\n" + "="*50 + "\n")

In [ ]:
### Quem são os 10 artistas que mais venderam?
### dica: você precisa juntar as tabelas invoice_items, tracks, albums e artists
print("--- Top 10 Artistas que Mais Venderam ---")
stmt6 = (
    select(Artists.Name, func.sum(InvoiceItems.Quantity).label('total_vendido'))
    .join(Albums, Artists.ArtistId == Albums.ArtistId)
    .join(Tracks, Albums.AlbumId == Tracks.AlbumId)
    .join(InvoiceItems, Tracks.TrackId == InvoiceItems.TrackId)
    .group_by(Artists.ArtistId, Artists.Name)
    .order_by(func.sum(InvoiceItems.Quantity).desc())
    .limit(10)
)

for nome_artista, total in session.execute(stmt6):
    print(f"Vendas totais: {total} | Artista: {nome_artista}")

session.close()